In [6]:
import pandas as pd
import os
import numpy as np

In [7]:
# set-up paths
base_path = r"C:\Users\ageglio\OneDrive - TRC\Documents\My EQuIS Work\Mdn01\Mosaic Hutchinson"
dt_location_path = os.path.join(base_path, "dt_location.xlsx")
dt_coordinate_path = os.path.join(base_path, "dt_coordinate.xlsx")
dt_measure_datum_path = os.path.join(base_path, "dt_measure_datum.xlsx")
# dt_well_segment_path = os.path.join(base_path, "dt_well_segment.xlsx")
dt_water_level_path = os.path.join(base_path, "dt_water_level.xlsx")
dt_well_path = os.path.join(base_path, "dt_well.xlsx")

# read raw data tables
dt_location = pd.read_excel(dt_location_path)
dt_coordinate = pd.read_excel(dt_coordinate_path)
dt_measure_datum = pd.read_excel(dt_measure_datum_path)
# dt_well_segment = pd.read_excel(dt_well_segment_path)
dt_water_level = pd.read_excel(dt_water_level_path)
dt_well = pd.read_excel(dt_well_path)

In [8]:
## pairing down the raw dt tables to only the columns we need
# dt_location
dt_location2 = dt_location[["sys_loc_code", "loc_type"]]

# dt_coordinate
dt_coordinatexy = dt_coordinate[dt_coordinate.coord_type_code=="SP_KS_S_NAD83_ft"][["sys_loc_code", "y_coord", "x_coord", "elev"]]
dt_coordinatell = dt_coordinate[dt_coordinate.coord_type_code=="LATLONG_NAD83"][["sys_loc_code", "y_coord", "x_coord"]
                        ].rename(columns={"y_coord": "latitude", "x_coord": "longitude"})

# dt_measure_datum
dt_measure_datum2 = dt_measure_datum[["sys_loc_code", "start_date", "datum_value"]]

# dt_water_level
dt_water_level2 = dt_water_level[["sys_loc_code", "measurement_date", "water_level_depth", "measured_depth_of_well"]
                                 ].dropna(subset=["water_level_depth", "measured_depth_of_well"], how="all")

# dt_well
dt_well2 = dt_well[["sys_loc_code", "geologic_unit_code", "installation_date", "depth_of_well"]]

# dt_well_segment
# dt_well_segment2 = dt_well_segment[["sys_loc_code", "start_depth", "end_depth", "material_type_code", "remark"]]

In [12]:
# 1. Standardize column names
df_water_level = dt_water_level2.rename(columns={'measurement_date': 'date'})
df_datum = dt_measure_datum2.rename(columns={'start_date': 'date'})

# 2. Use an OUTER MERGE on location and date
# This combines same-timestamp rows and keeps unique ones separate
df_merged = pd.merge(df_water_level, df_datum, on=['sys_loc_code', 'date'], how='outer')

# 3. Sort chronologically
df_merged = df_merged.sort_values(by=['sys_loc_code', 'date'])

# 4. Forward fill the datum
# This ensures measurements between datum changes use the most recent value
df_merged['datum_value'] = df_merged.groupby('sys_loc_code')['datum_value'].ffill()

# 5. Handle the well depth backfill/forward fill as before
df_merged['measured_depth_of_well'] = df_merged.groupby('sys_loc_code')['measured_depth_of_well'].ffill()

# 6. Calculate elevations
df_merged['water_level_elevation'] = df_merged['datum_value'] - df_merged['water_level_depth']
df_merged['bottom_elevation'] = df_merged['datum_value'] - df_merged['measured_depth_of_well']

# 7. Merge with location and coordinate data
df_merged2 = (df_merged
              .merge(dt_coordinatexy, on="sys_loc_code", how="left")
              .merge(dt_coordinatell, on="sys_loc_code", how="left"))


# convert date column to datetime format that excel will recognize as a date
df_merged2["date"] = pd.to_datetime(df_merged2["date"], errors='coerce').dt.date
df_merged2.to_excel(os.path.join(base_path, "mosaic_hutchinson_water_levels.xlsx"), index=False)

In [11]:
df_merged2[df_merged2.sys_loc_code == 'MW-18S']

,sys_loc_code,date,water_level_depth,measured_depth_of_well,datum_value,water_level_elevation,bottom_elevation,y_coord,x_coord,elev,latitude,longitude
1283,MW-18S,2007-01-01,14.92,28.18,1525.92,1511.00,1497.74,1.816949e+06,1.485114e+06,NaN,38.050975,-97.89995
1284,MW-18S,2007-04-01,11.61,28.18,1525.92,1514.31,1497.74,1.816949e+06,1.485114e+06,NaN,38.050975,-97.89995
1285,MW-18S,2007-07-01,12.93,28.18,1525.92,1512.99,1497.74,1.816949e+06,1.485114e+06,NaN,38.050975,-97.89995
1286,MW-18S,2007-10-01,13.48,28.18,1525.92,1512.44,1497.74,1.816949e+06,1.485114e+06,NaN,38.050975,-97.89995
1287,MW-18S,2008-03-10,13.38,28.18,1525.92,1512.54,1497.74,1.816949e+06,1.485114e+06,NaN,38.050975,-97.89995
1288,MW-18S,2008-06-16,12.33,28.18,1525.92,1513.59,1497.74,1.816949e+06,1.485114e+06,NaN,38.050975,-97.89995
1289,MW-18S,2008-09-15,12.56,28.18,1525.92,1513.36,1497.74,1.816949e+06,1.485114e+06,NaN,38.050975,-97.89995
1290,MW-18S,2009-02-09,13.25,28.18,1525.92,1512.67,1497.74,1.816949e+06,1.485114e+06,NaN,38.050975,-97.89995
1291,MW-18S,2009-04-27,12.78,28.18,1525.92,1513.14,1497.74,1.816949e+06,1.485114e+06,NaN,38.050975,-97.89995
1292,MW-18S,2009-09-28,12.93,28.18,1525.92,1512.99,1497.74,1.816949e+06,1.485114e+06,NaN,38.050975,-97.89995
